In [4]:
import json
import urllib.parse
from pathlib import Path
from typing import Any, Sequence

In [2]:
from pathlib import Path
import json
import h5py
import re  # for regex

import networkx as nx
from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter
from sklearn.metrics import pairwise_distances

from copy import deepcopy
import itertools
from tqdm import tqdm

import random
import numpy as np
import pandas as pd

import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib as mpl

from efish_em import AnalysisCode as efish #only works after the efish_em package has been installed as per README

In [70]:
_BASE_URL: str = "https://neuroglancer-demo.appspot.com/#!"

DEFAULT_SEG_SOURCE: dict = {
    "url": (
        "gs://efish-public/roi450um_published_260420/|neuroglancer-precomputed:"
    ),
    "subsources": {"default": True, "meshes": True, "bounds": True},
    "enableDefaultSubsources": False,
}

DEFAULT_IMG_SOURCE: dict = {
    "type": "image",
    "source": {
        "url": "gs://efish-public/roi450um_aligned/|neuroglancer-precomputed:",
        "subsources": {"default": True},
        "enableDefaultSubsources": False,
    },
    "tab": "source",
    "name": "em-8nm",
}

DEFAULT_DIMENSIONS: dict = {
    "x": [1.6e-8, "m"],
    "y": [1.6e-8, "m"],
    "z": [3e-8, "m"],
}

# ---------------------------------------------------------------------------
# Default camera / view
# ---------------------------------------------------------------------------

DEFAULT_POSITION: tuple[float, float, float] = (
    91798.328125,
    54608.82421875,
    6048.93359375
)
DEFAULT_CROSS_SECTION_SCALE: float = 13.5
DEFAULT_PROJECTION_SCALE: float = 45628.483272660975
DEFAULT_PROJECTION_ORIENTATION: tuple[float, float, float] = (
    -0.057177431881427765,
    0.0019532477017492056,
    -0.000014164616004563868,
    0.998362123966217
)
DEFAULT_BACKGROUND: str = "#ffffff"

In [37]:
def clio_state_to_url(state: dict) -> str:
    """Serialize a Neuroglancer state dict into a Clio URL.

    Parameters
    ----------
    state : dict
        A Neuroglancer-compatible JSON state dictionary.

    Returns
    -------
    str
        A full ``https://clio-ng.janelia.org/#!...`` URL.
    """
    payload = json.dumps(state, separators=(",", ":"))
    return _BASE_URL + urllib.parse.quote(payload, safe="")


# ---------------------------------------------------------------------------
# Layer builders
# ---------------------------------------------------------------------------

def build_seg_layer(
    name: str,
    ids: Sequence[int],
    *,
    seg_source: dict | None = None,
    color: str,
    object_alpha: float
) -> dict:
    """Build a Neuroglancer segmentation layer dict.

    Parameters
    ----------
    name : str
        Display name for the layer.
    ids : sequence of int
        Body IDs to include in the layer.
    seg_source : dict or None
        A Neuroglancer source object.  If *None* a sensible default
        for the current fish2 DVID node is used.
    color : str
        Hex colour string (e.g. ``"#ff0000"``).
    alpha : float
        transparency for 3D mesh rendering

    Returns
    -------
    dict
        A Neuroglancer layer specification ready to be added to the
        ``"layers"`` list of a state dict.
    """
    if seg_source is None:
        seg_source = {
            "url": (
                "dvid://https://fishemf-cleaving.janelia.org"
                "/f3fc92:master/segmentation"
                "?dvid-service="
                "https://ngsupport-bmcp5imp6q-uk.a.run.app"
            ),
            "subsources": {"default": True, "meshes": True},
            "enableDefaultSubsources": False,
        }
    segments = [str(int(i)) for i in ids] if ids else []
    return {
        "type": "segmentation",
        "source": seg_source,
        "tab": "rendering",
        "name": name,
        "segments": segments,
        "segmentDefaultColor": color,
        "objectAlpha": object_alpha,
        "toolBindings": {"Q": "selectSegments"},
    }


def build_annotation_layer(
    name: str,
    points_xyz_nm: Sequence[Sequence[float]],
    color: str = "#ff0000",
    tool: str = "annotatePoint",
    id_prefix: str | None = None,
    voxel_size_nm: Sequence[float] = (8, 8, 30),
) -> dict:
    """Build a Neuroglancer point-annotation layer dict.

    Parameters
    ----------
    name : str
        Display name for the annotation layer.
    points_xyz_nm : sequence of (x, y, z)
        Point coordinates in **nanometres** (as returned by neuPrint).
    color : str
        Hex colour for the annotations.
    tool : str
        Neuroglancer annotation tool (default ``"annotatePoint"``).
    id_prefix : str or None
        Prefix for annotation IDs.  Defaults to *name*.
    voxel_size_nm : sequence of float
        Voxel dimensions in nm used to convert coordinates.

    Returns
    -------
    dict
        A Neuroglancer annotation-layer specification.

    Notes
    -----
    Neuroglancer only allows one colour per annotation layer.
    """
    ann_list: list[dict] = []
    for i, p in enumerate(points_xyz_nm):
        if p is None:
            continue
        if any(
            v is None or (isinstance(v, float) and np.isnan(v))
            for v in p
        ):
            continue

        uid = f"{id_prefix or name}_{i}"
        pt = [float(p_ / v) for p_, v in zip(p, voxel_size_nm)]
        ann_list.append(
            {
                "id": uid,
                "type": "point",
                "point": pt,
                "description": name,
            }
        )

    return {
        "type": "annotation",
        "tab": "annotations",
        "name": name,
        "tool": tool,
        "annotationColor": color,
        "annotations": ann_list,
    }

In [6]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
vx_sizes = [16, 16, 30]

In [7]:
cell_colors = efish.color_palette('cell')
structure_colors = efish.color_palette('structure')

## Load reconstruction files

In [8]:
nodefiles = efish.get_cell_filepaths(dirpath)

# cell types for all files in directory

From file (see CellTyping.ipynb for how the dataframe was created)

In [9]:
df_type = pd.read_csv(dirpath.parent / 'data_processed_published/df_type_auto_typed.csv')

In [71]:
layers = [DEFAULT_IMG_SOURCE]

In [72]:
tab_names = ['ON','OFF','MG2','MG1','EAF']
for i,ctype in enumerate(['lf','lg','mg2','mg1','aff']):

    ids = list(df_type.query("cell_type == @ctype")['id'].unique())
    
    seg_layer = build_seg_layer(
        name= tab_names[i],
        ids=ids,
        seg_source=DEFAULT_SEG_SOURCE,
        color=cell_colors[ctype],
        object_alpha= 1
    )
    layers.append(seg_layer)


## Synapses dataframe

In [22]:
df_syn = pd.read_csv(dirpath.parent / 'data_processed_published/df_postsyn.csv')

## add cell type to df_syn

In [23]:
for i,r in df_syn.iterrows():
    try:
        df_syn.loc[i,'pre_type'] =df_type[df_type['id'].isin([r['pre']])].cell_type.values[0]
        df_syn.loc[i,'post_type']=df_type[df_type['id'].isin([r['post']])].cell_type.values[0]
    except:
        print(r['pre'],r['post'])
        continue

In [73]:
for ctype in ['mg1','mg2']:
    mask = df_syn['pre_type'].isin([ctype])
    
    syn = build_annotation_layer(
        name= f'{ctype}_syn_targets',
        points_xyz_nm= df_syn[mask][['x','y','z']].values,
        color = cell_colors[ctype],
        tool = "annotatePoint",
        voxel_size_nm= (16, 16, 30)
    )
    layers.append(syn)

In [74]:



state = {
    "title": 'efish_em_ELL',
    "dimensions": {
    "x": [
      1.6e-8,
      "m"
    ],
    "y": [
      1.6e-8,
      "m"
    ],
    "z": [
      3e-8,
      "m"
    ]
    },
    "position": [
    15438.5,
    16532.5,
    885.5
    ],
    "crossSectionScale": 46.15475549720696,
    "projectionScale": 35731.54279325757,

    "projectionBackgroundColor": '#878787',
    "layers": layers,
    "showSlices": False,
    "gpuMemoryLimit": 2_000_000_000,
    "systemMemoryLimit": 4_000_000_000,
    "selectedLayer": (
        {"layer": layers[-1]["name"]}
    ),
    "layout": "3d"
}

In [75]:
print(clio_state_to_url(state))

https://neuroglancer-demo.appspot.com/#!%7B%22title%22%3A%22efish_em_ELL%22%2C%22dimensions%22%3A%7B%22x%22%3A%5B1.6e-08%2C%22m%22%5D%2C%22y%22%3A%5B1.6e-08%2C%22m%22%5D%2C%22z%22%3A%5B3e-08%2C%22m%22%5D%7D%2C%22position%22%3A%5B15438.5%2C16532.5%2C885.5%5D%2C%22crossSectionScale%22%3A46.15475549720696%2C%22projectionScale%22%3A35731.54279325757%2C%22projectionBackgroundColor%22%3A%22%23878787%22%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22source%22%3A%7B%22url%22%3A%22gs%3A%2F%2Fefish-public%2Froi450um_aligned%2F%7Cneuroglancer-precomputed%3A%22%2C%22subsources%22%3A%7B%22default%22%3Atrue%7D%2C%22enableDefaultSubsources%22%3Afalse%7D%2C%22tab%22%3A%22source%22%2C%22name%22%3A%22em-8nm%22%7D%2C%7B%22type%22%3A%22segmentation%22%2C%22source%22%3A%7B%22url%22%3A%22gs%3A%2F%2Fefish-public%2Froi450um_published_260420%2F%7Cneuroglancer-precomputed%3A%22%2C%22subsources%22%3A%7B%22default%22%3Atrue%2C%22meshes%22%3Atrue%2C%22bounds%22%3Atrue%7D%2C%22enableDefaultSubsources%22%3Afal

In [77]:
with open('/Users/kperks/Downloads/Proofread_MG_Output_MGsyn.json', 'w') as f:
    json.dump(state, f, indent=4)